# # llm
#
# > One call shape for every model: `ask(prompt, sp) -> str`.

In [ ]:
#| default_exp llm

In [ ]:
from fastcore.test import *

In [ ]:
#| export
import json, os, re
from fastcore.utils import *

# The tutor needs only one thing from a model: text for a prompt. Each surface gives its own `ask`:
#
# - the web app and the notebook use `rishi_ask`, a new rishi `Chat` for each call, on `$ABHYASA_MODEL` (default `claude/sonnet`);
# - ramabana gives `agent.oneshot`, so ramabana's routing applies;
# - tests give `Fake`.

In [ ]:
#| export
DFLT_MODEL = 'claude/sonnet'

def rishi_ask(model:str=None):
    "An `ask(prompt, sp='')` on a rishi `Chat` for `model`, else `$ABHYASA_MODEL`, else `DFLT_MODEL`."
    from rishi import Chat, resp_text
    model = model or os.getenv('ABHYASA_MODEL') or DFLT_MODEL
    def ask(prompt, sp=''):
        chat = Chat(model, sp=sp)
        try: return resp_text(chat(prompt))
        finally: chat.close()
    ask.model = model
    return ask

In [ ]:
#| export
def parse_json(s:str):
    "The first JSON object or array in `s`, also inside a ``` fence. Raises `ValueError` when there is none."
    s = s.strip()
    if (m := re.search(r'```(?:json)?\s*(.*?)```', s, re.S)): s = m.group(1).strip()
    try: return json.loads(s)
    except json.JSONDecodeError: pass
    for o, cl in ('{}', '[]'):
        i, j = s.find(o), s.rfind(cl)
        if 0 <= i < j:
            try: return json.loads(s[i:j+1])
            except json.JSONDecodeError: continue
    raise ValueError(f'no JSON in the reply: {s[:200]!r}')

def ask_json(ask, prompt:str, sp:str='', tries:int=2):
    "Ask for JSON and parse it; on a bad reply, ask once more with the parse error."
    p = prompt + '\n\nReply with only one JSON value and no other text.'
    for i in range(tries):
        out = ask(p, sp)
        try: return parse_json(out)
        except ValueError as e:
            if i == tries - 1: raise
            p = f'{prompt}\n\nYour last reply was not valid JSON ({e}). Reply with only one JSON value.'

In [ ]:
test_eq(parse_json('```json\n{"a": 1}\n```'), {'a': 1})
test_eq(parse_json('Here it is: {"a": [1, 2]} thanks'), {'a': [1, 2]})
test_eq(parse_json('[{"x": 1}]'), [{'x': 1}])
test_fail(lambda: parse_json('nothing'), contains='no JSON')

# `Fake` answers from a list of rules: the first rule whose key is in the prompt gives the reply. A reply that is a dict
# or a list is sent as JSON. It keeps every call in `calls`.

In [ ]:
#| export
class Fake:
    "A stand-in model for tests: `Fake({'grade': {...}, 'question': {...}})`."
    def __init__(self, rules:dict=None, dflt='{}'): self.rules, self.dflt, self.calls = rules or {}, dflt, []
    def __call__(self, prompt, sp=''):
        self.calls.append((prompt, sp))
        r = next((v for k, v in self.rules.items() if k in prompt or k in sp), self.dflt)
        if callable(r): r = r(prompt, sp)
        return r if isinstance(r, str) else json.dumps(r)

In [ ]:
f = Fake({'GRADE': dict(correct=True)})
test_eq(ask_json(f, 'GRADE this'), dict(correct=True))
test_eq(len(f.calls), 1)
bad = Fake(dict(X=lambda p, sp: 'oops' if 'not valid' not in p else '{"ok": 1}'))
test_eq(ask_json(bad, 'X'), {'ok': 1})

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()